<a href="https://colab.research.google.com/github/yashvi1721/16_mobilecomputing/blob/main/sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import json
import io

# ==========================================
# 1. LOAD DATA & RULES
# ==========================================
# The debug output showed columns were lumped into one string.
# We will use sep=None to sniff, and if that fails, manually split.
df = pd.read_csv("/content/sample_data/FLS_KPI_Dummy_Data.csv - Sheet1.csv", sep=None, engine='python')

# If it still loads as one column, force split it
if len(df.columns) == 1:
    col_name = df.columns[0]
    df = df[col_name].str.split(',', expand=True)
    df.columns = col_name.split(',')

# Strip potential whitespace and normalize column names
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(' ', '_')

with open("/content/sample_data/dec.json", "r") as f:
    reward_rules = json.load(f)

# Pivot data to have one row per employee
if 'KPI_Value' in df.columns and 'KPI_Parameter' in df.columns:
    pivot_df = (
        df.pivot_table(
            index=["Emp_Code"],
            columns="KPI_Parameter",
            values="KPI_Value",
            aggfunc="first"
        ).reset_index()
    )
    pivot_df.columns.name = None
    pivot_df["Emp_Code"] = pd.to_numeric(pivot_df["Emp_Code"], errors="coerce").fillna(0).astype(int).astype(str)
else:
    print("Error: Columns not found. Available:", df.columns.tolist())

# Rest of calculation logic...
def evaluate_condition(condition, emp):
    metric = condition.get("metric")
    operator = condition.get("operator")
    value = condition.get("value")
    emp_val = emp.get(metric)
    if emp_val is None: return False
    try:
        emp_val = float(emp_val) if str(emp_val).strip() not in ["Pass", "Fail"] else str(emp_val).strip()
    except:
        return False
    if operator == ">=": return emp_val >= value
    if operator == ">": return emp_val > value
    if operator == "<=": return emp_val <= value
    if operator == "<": return emp_val < value
    if operator == "==": return emp_val == value
    return False

def calculate_derived_inputs(emp):
    derived = {}
    sp_1_15 = float(emp.get("SP_Activated_1_15", 0) or 0)
    sp_16_31 = float(emp.get("SP_Activated_16_31", 0) or 0)
    sp_alloc = float(emp.get("SP_Allocated", 1) or 1)
    nop_1_15 = float(emp.get("NOP_1_15", 0) or 0)
    nop_16_31 = float(emp.get("NOP_16_31", 0) or 0)
    derived["sp_activated_total"] = sp_1_15 + sp_16_31
    derived["sp_activation_percent"] = (derived["sp_activated_total"] / sp_alloc * 100) if sp_alloc > 0 else 0
    derived["nop_total"] = nop_1_15 + nop_16_31
    derived["nop_1_15"] = nop_1_15
    derived["nop_16_31"] = nop_16_31
    derived["sp_1_15"] = sp_1_15
    derived["sp_16_31"] = sp_16_31
    return derived

def calc_kpi_reward(kpi, emp, day, derived):
    kpi_name = kpi.get("name")
    kpi_type = kpi.get("type")
    details = {"kpi_name": kpi_name, "reward": 0, "calc_text": "", "value_display": ""}
    if kpi_type == "binary":
        conditions = kpi.get("conditions", [])
        all_pass = all([evaluate_condition(c, emp) for c in conditions])
        details["reward"] = kpi.get("payout", {}).get("pass", 0) if all_pass else 0
        details["calc_text"] = "All conditions met" if all_pass else "Conditions failed"
        details["value_display"] = "Checked"
        return details
    if kpi_type == "categorical":
        val = str(emp.get(kpi.get("metric"), ""))
        details["reward"] = kpi.get("payout", {}).get(val, 0)
        details["calc_text"] = f"Status: {val}"
        details["value_display"] = val
        return details
    if kpi_type == "slab":
        ctc_alloc = float(emp.get("CTC_Allocated", 0) or 0)
        ctc_conn = float(emp.get("CTC_Connected", 0) or 0)
        conn_pct = (ctc_conn / ctc_alloc * 100) if ctc_alloc > 0 else 0
        details["value_display"] = f"Allocated: {ctc_alloc}, Connected: {ctc_conn} ({conn_pct:.2f}%)"
        matched_slab = None
        slabs = kpi.get("slabs", [])
        for slab in slabs:
            crit = slab.get("criteria", "").lower().replace("ctc_allocated", str(ctc_alloc)).replace("connect_percent", str(conn_pct))
            try:
                if eval(crit): matched_slab = slab; break
            except: pass
        if matched_slab: details["reward"] = matched_slab.get("payout", 0); details["calc_text"] = f"Slab matched: {matched_slab.get('criteria')}"
        else: details["reward"] = 0; details["calc_text"] = "No slab matched"
        return details
    if kpi_type in ["slab_with_per_unit", "per_unit"]:
        slabs = kpi.get("slabs", [])
        matched_slab = None
        if kpi_name == "SP_Activation":
            details["value_display"] = f"SP% = {derived['sp_activation_percent']:.2f}%"
            for slab in slabs:
                crit = slab.get("criteria").lower().replace("sp_activation_percent", str(derived["sp_activation_percent"]))
                try:
                    if eval(crit): matched_slab = slab; break
                except: pass
        elif kpi_name == "NOP":
            details["value_display"] = f"NOP Total = {derived['nop_total']}"
            for slab in slabs:
                crit = slab.get("criteria").lower().replace("nop_total", str(derived["nop_total"]))
                try:
                    if eval(crit): matched_slab = slab; break
                except: pass
        else: matched_slab = {"payout": kpi.get("payout")}
        if matched_slab:
            payout = matched_slab.get("payout", {})
            rate_1, rate_2 = float(payout.get("1-15", 0) or 0), float(payout.get("16-31", 0) or 0)
            if day <= 15:
                reward = (derived["sp_1_15"] if kpi_name == "SP_Activation" else derived["nop_1_15"]) * rate_1
            else:
                reward = ((derived["sp_1_15"] * rate_1) + (derived["sp_16_31"] * rate_2)) if kpi_name == "SP_Activation" else ((derived["nop_1_15"] * rate_1) + (derived["nop_16_31"] * rate_2))
            cap = kpi.get("max_payout_cap")
            if cap and reward > cap: reward = cap
            details["reward"] = reward
        return details
    return details

def calculate_full_incentive(emp_code, day=16, override_data=None):
    if 'pivot_df' not in globals(): return None
    emp_row_df = pivot_df[pivot_df["Emp_Code"] == str(emp_code)]
    if emp_row_df.empty: return None
    emp_row = emp_row_df.iloc[0].to_dict()
    if override_data: emp_row.update(override_data)
    vintage = float(emp_row.get("Vintage", 0) or 0)
    vintage_cat = "Vintage_0_6_Months" if vintage <= 180 else "Vintage_6_Plus_Months"
    rules = reward_rules["BancaBooster_Dec25"][vintage_cat]
    all_kpis = rules.get("rule_based_kpis", []) + rules.get("calculated_kpis", [])
    derived = calculate_derived_inputs(emp_row)
    kpi_results = [calc_kpi_reward(kpi, emp_row, day, derived) for kpi in all_kpis]
    total_earned = sum(res["reward"] for res in kpi_results)
    gate_passed = all([evaluate_condition(c, emp_row) for c in rules.get("overall_gate", {}).get("criteria", [])])
    return {"emp_code": emp_code, "kpi_results": kpi_results, "final_payable": total_earned if gate_passed else 0, "gate_passed": gate_passed, "total_earned": total_earned, "vintage_category": vintage_cat}

# Runner
emp_no = input("Enter Employee Number: ")
res = calculate_full_incentive(emp_no)
if res: print(f"Final Payable for Emp {emp_no}: {res['final_payable']}")
else: print("Employee not found or data error.")

Columns found: ['Emp_Code,Channel,Sub_Channel,Agent,Month,Year,KPI_Parameter,KPI_Value']
Error: Required columns not found. Please check CSV format.
Enter Employee Number: 2
